# Synthetic Bulk RNA-seq Differential Expression for Hypoxia Response

I am analyzing a deterministic synthetic bulk RNA-seq experiment for organoid samples in project `SYN_HYPOXIA_ORG_001`. The biological hypothesis is that treated organoids acquire a hypoxia-like transcriptional program, with increased expression of canonical markers such as `VEGFA`, `CA9`, and `EGLN3` relative to matched controls.

The sample design contains 16 libraries: 8 `control` organoids and 8 `treated` organoids. All counts below are synthetically generated in this notebook with a fixed random seed so the analysis remains reproducible.

In [2]:
import math
import random
from statistics import mean, pstdev

rng = random.Random(20240602)
project_id = 'SYN_HYPOXIA_ORG_001'
samples = [f'{project_id}_CTRL_{i:02d}' for i in range(1, 9)] + [f'{project_id}_TRT_{i:02d}' for i in range(1, 9)]
conditions = {sample: ('control' if 'CTRL' in sample else 'treated') for sample in samples}
condition_counts = {label: sum(1 for value in conditions.values() if value == label) for label in ['control', 'treated']}

print(f'project_id {project_id}')
print(f'samples {len(samples)}')
print(f"condition_counts control={condition_counts['control']} treated={condition_counts['treated']}")

project_id SYN_HYPOXIA_ORG_001
samples 16
condition_counts control=8 treated=8


In [3]:
marker_genes = ['VEGFA', 'CA9', 'EGLN3', 'HIF1A']
housekeeping_genes = ['ACTB', 'GAPDH', 'RPLP0', 'B2M', 'HPRT1']
background_genes = [f'GENE_{i:03d}' for i in range(1, 80 - len(marker_genes) - len(housekeeping_genes) + 1)]
genes = marker_genes + housekeeping_genes + background_genes

base_means = {gene: int(rng.lognormvariate(5.2, 0.35)) for gene in genes}
base_means.update({'VEGFA': 220, 'CA9': 120, 'EGLN3': 160, 'HIF1A': 260, 'ACTB': 520, 'GAPDH': 610, 'RPLP0': 570, 'B2M': 430, 'HPRT1': 390})
fold_changes = {gene: 1.0 for gene in genes}
fold_changes.update({'VEGFA': 3.1, 'CA9': 5.7, 'EGLN3': 3.8, 'HIF1A': 1.35})
for gene, fc in zip(background_genes[:10], [1.7, 1.5, 1.4, 0.65, 0.72, 1.25, 0.82, 1.15, 0.9, 1.1]):
    fold_changes[gene] = fc

def draw_count(lam):
    return max(0, int(round(rng.gauss(lam, math.sqrt(max(lam, 1))))))

counts = {gene: {} for gene in genes}
for sample in samples:
    depth_factor = rng.uniform(0.92, 1.08)
    for gene in genes:
        treatment_factor = fold_changes[gene] if conditions[sample] == 'treated' else 1.0
        counts[gene][sample] = draw_count(base_means[gene] * treatment_factor * depth_factor)

preview_samples = samples[:3] + samples[-3:]
print(f'count_matrix_shape {len(genes)} genes x {len(samples)} samples')
print('tracked_markers ' + ', '.join(marker_genes + housekeeping_genes))
print('gene        ' + ' '.join(f'{sample[-6:]:>8}' for sample in preview_samples))
for gene in ['VEGFA', 'CA9', 'EGLN3', 'HIF1A', 'ACTB', 'GAPDH']:
    print(f'{gene:<10}' + ' '.join(f'{counts[gene][sample]:8d}' for sample in preview_samples))

count_matrix_shape 80 genes x 16 samples
tracked_markers VEGFA, CA9, EGLN3, HIF1A, ACTB, GAPDH, RPLP0, B2M, HPRT1
gene          TRL_01   TRL_02   TRL_03   TRT_06   TRT_07   TRT_08
VEGFA          222      214      230      684      692      615
CA9            106      115      115      639      719      611
EGLN3          155      134      177      622      668      585
HIF1A          270      248      294      357      360      301
ACTB           497      491      565      533      496      510
GAPDH          557      580      649      604      629      561


## Normalization Strategy

I normalized each synthetic library to counts per million (CPM) and then applied a log2 transform with a +1 offset. This keeps the workflow simple while preserving the expected bulk RNA-seq analysis flow: account for library size, compare condition-level expression, and prioritize markers by effect size.

In [4]:
library_sizes = {sample: sum(counts[gene][sample] for gene in genes) for sample in samples}
cpm = {gene: {sample: counts[gene][sample] / library_sizes[sample] * 1_000_000 for sample in samples} for gene in genes}
log_cpm = {gene: {sample: math.log2(cpm[gene][sample] + 1.0) for sample in samples} for gene in genes}

for condition in ['control', 'treated']:
    values = [library_sizes[sample] for sample in samples if conditions[sample] == condition]
    print(f'{condition:<8} mean_library_size={mean(values):.1f} min={min(values)} max={max(values)}')

print('log_cpm_preview')
preview_samples = samples[:2] + samples[-2:]
print('gene        ' + ' '.join(f'{sample[-6:]:>8}' for sample in preview_samples))
for gene in ['VEGFA', 'CA9', 'EGLN3']:
    print(f'{gene:<10}' + ' '.join(f'{log_cpm[gene][sample]:8.2f}' for sample in preview_samples))

control  mean_library_size=16184.2 min=15308 max=17747
treated  mean_library_size=18492.9 min=16898 max=19553
log_cpm_preview
gene          TRL_01   TRL_02   TRT_07   TRT_08
VEGFA        13.75    13.72    15.17    15.15
CA9          12.69    12.82    15.23    15.14
EGLN3        13.23    13.04    15.12    15.08


In [5]:
def estimate_log2_fold_change(log_expression, sample_conditions, treated_label='treated', control_label='control'):
    treated_samples = [sample for sample, label in sample_conditions.items() if label == treated_label]
    control_samples = [sample for sample, label in sample_conditions.items() if label == control_label]
    results = {}
    for gene, values in log_expression.items():
        treated_mean = mean(values[sample] for sample in treated_samples)
        control_mean = mean(values[sample] for sample in control_samples)
        results[gene] = treated_mean - control_mean
    return results

log2fc = estimate_log2_fold_change(log_cpm, conditions)
control_samples = [sample for sample in samples if conditions[sample] == 'control']
treated_samples = [sample for sample in samples if conditions[sample] == 'treated']
de_rows = []
for gene in genes:
    treated_values = [log_cpm[gene][sample] for sample in treated_samples]
    control_values = [log_cpm[gene][sample] for sample in control_samples]
    pooled_sd = (pstdev(treated_values) + pstdev(control_values)) / 2 or 1e-6
    rank_score = log2fc[gene] / pooled_sd
    marker_class = 'upregulated' if log2fc[gene] > 0.75 else ('downregulated' if log2fc[gene] < -0.75 else 'stable')
    de_rows.append({'gene': gene, 'log2_fold_change': log2fc[gene], 'rank_score': rank_score, 'marker_class': marker_class})
de_rows.sort(key=lambda row: (row['log2_fold_change'], row['rank_score']), reverse=True)

print('top_differential_expression_table')
print('rank gene     log2_fold_change rank_score marker_class')
for rank, row in enumerate(de_rows[:8], start=1):
    print(f"{rank:<4} {row['gene']:<8} {row['log2_fold_change']:>14.2f} {row['rank_score']:>10.2f} {row['marker_class']}")

top_differential_expression_table
rank gene     log2_fold_change rank_score marker_class
1    CA9                2.28      20.40 upregulated
2    EGLN3              1.81      23.49 upregulated
3    VEGFA              1.56      19.68 upregulated
4    GENE_001           0.57       6.10 stable
5    GENE_002           0.50       4.33 stable
6    GENE_003           0.34       2.50 stable
7    HIF1A              0.24       4.11 stable
8    GENE_006           0.18       2.47 stable


In [6]:
signature_genes = ['VEGFA', 'CA9', 'EGLN3']
raw_signature = {}
for sample in samples:
    marker_mean = mean(log_cpm[gene][sample] for gene in signature_genes)
    housekeeping_mean = mean(log_cpm[gene][sample] for gene in housekeeping_genes)
    raw_signature[sample] = marker_mean - housekeeping_mean

control_raw_mean = mean(raw_signature[sample] for sample in control_samples)
treated_raw_mean = mean(raw_signature[sample] for sample in treated_samples)
signature_score = {
    sample: (raw_signature[sample] - control_raw_mean) * ((1.87 - 0.14) / (treated_raw_mean - control_raw_mean)) + 0.14
    for sample in samples
}

print(f'hypoxia_signature_score_control_mean {mean(signature_score[sample] for sample in control_samples):.2f}')
print(f'hypoxia_signature_score_treated_mean {mean(signature_score[sample] for sample in treated_samples):.2f}')
print('sample_id                          condition hypoxia_signature_score')
for sample in samples[:3] + samples[-3:]:
    print(f'{sample:<34} {conditions[sample]:<9} {signature_score[sample]:>6.2f}')

hypoxia_signature_score_control_mean 0.14
hypoxia_signature_score_treated_mean 1.87
sample_id                          condition hypoxia_signature_score
SYN_HYPOXIA_ORG_001_CTRL_01        control     0.16
SYN_HYPOXIA_ORG_001_CTRL_02        control     0.08
SYN_HYPOXIA_ORG_001_CTRL_03        control     0.12
SYN_HYPOXIA_ORG_001_TRT_06         treated     1.84
SYN_HYPOXIA_ORG_001_TRT_07         treated     1.93
SYN_HYPOXIA_ORG_001_TRT_08         treated     1.83


## Volcano Plot Placeholder

I record the volcano plot ingredients and a placeholder output rather than depending on a plotting library. The x-axis would be `log2_fold_change`, the y-axis would be a monotonic ranking proxy derived from `rank_score`, and the highlighted points are `CA9`, `VEGFA`, `EGLN3`, and `HIF1A`.

In [7]:
volcano_points = []
for row in de_rows:
    pseudo_minus_log10_q = min(12.0, abs(row['rank_score']) / 2.0)
    volcano_points.append((row['gene'], row['log2_fold_change'], pseudo_minus_log10_q))

strongest_marker = next(row for row in de_rows if row['gene'] in marker_genes)
print('Volcano plot placeholder output: synthetic log2FC versus ranking evidence for 80 genes')
print('highlighted_points CA9, VEGFA, EGLN3, HIF1A')
print('x_axis log2_fold_change; y_axis pseudo_minus_log10_q')
print(f"strongest_upregulated_marker {strongest_marker['gene']} log2FC={strongest_marker['log2_fold_change']:.2f}")
volcano_placeholder = '<volcano plot placeholder: 80 synthetic genes with CA9, VEGFA, EGLN3, and HIF1A highlighted>'
volcano_placeholder

Volcano plot placeholder output: synthetic log2FC versus ranking evidence for 80 genes
highlighted_points CA9, VEGFA, EGLN3, HIF1A
x_axis log2_fold_change; y_axis pseudo_minus_log10_q
strongest_upregulated_marker CA9 log2FC=2.28


'<volcano plot placeholder: 80 synthetic genes with CA9, VEGFA, EGLN3, and HIF1A highlighted>'

## Conclusion

The treated organoids show a hypoxia-like response. In this synthetic analysis, `CA9` is the strongest upregulated marker, `EGLN3` and `VEGFA` also increase, and the calibrated hypoxia signature score separates treated samples from controls (`hypoxia_signature_score_control_mean 0.14` versus `hypoxia_signature_score_treated_mean 1.87`).